> **_This is the continuation of the model training on the same dataset, but this time I will be using Decision Tree and Random Forest algorithms._**

# Decision Tree and Random Forest
- Decision Tree is a supervised learning algorithm used for classification and regression tasks. It works by recursively splitting the data into subsets based on the feature that provides the best separation of the classes or values. The resulting tree structure can be easily interpreted and visualized.
- Random Forest is an ensemble learning method that combines multiple decision trees to improve the accuracy and robustness of the model. It works by creating multiple decision trees using random subsets of the data and features, and then aggregating their predictions to make a final decision. Random Forest can handle large datasets and is less prone to overfitting compared to a single decision tree.


# Decision Tree

**Decision Tree** is a non-parametric supervised learning algorithm used for classification and regression tasks. It works by recursively splitting the data into subsets based on the feature that provides the best separation of the classes or values. The resulting tree structure can be easily interpreted and visualized, making it a popular choice for many machine learning problems.
> Non-parametric means that the model does not make any assumptions like linearity or normality  about the underlying distribution of the data. This allows decision trees to be flexible and adaptable to a wide range of datasets, as they can capture complex relationships between features without being constrained by a specific functional form.
- **Core Components**
The hierarchical structure of a decision tree mimics the way humans make decisions and consists of the following core components:
1. **Root Node**: The topmost node of the tree that represents the entire dataset. It is the starting point for making predictions.
2. **Internal (Decision) Nodes**: These nodes represent the features or attributes used to split the data. Each internal node corresponds to a specific feature and contains a decision rule that determines how the data is split based on that feature.
3. **Branches**: These are the connections between nodes that represent the outcome of a decision rule. Each branch corresponds to a specific value or range of values for the feature being evaluated at the internal node.
4. **Leaf (Terminal) Nodes**: These are the terminal nodes of the tree that represent the final output or prediction. In classification tasks, leaf nodes correspond to class labels, while in regression tasks, they correspond to continuous values.
![Decision Tree Structure](../data/dt_workflow.png)
**How It Works** \
Decision trees are built using a top-down, greedy approach called recursive partitioning:
- **Feature Selection**: The algorithm evaluates different features to find the one that best separates the data into "pure" groups (where most items belong to the same class).
- **Splitting Criteria**: To determine the best split, models typically use mathematical metrics like:
  - **Gini Impurity**: Measures how often a randomly chosen element would be incorrectly identified.
  - **Entropy/Information Gain**: Measures the reduction in uncertainty after a split.
- **Recursion**: This process repeats for each sub-group until the nodes are pure or a stopping condition (like maximum tree depth) is met.

**Key Advantages and Use Cases** \
Decision trees are widely used because they are white-box models—their logic is transparent and easy to visualize.
- **Versatility**: They handle both numerical and categorical data with minimal preprocessing (no scaling required).
- **Common Applications**:
  - **Finance**: Loan approval based on credit score and income.
  - **Healthcare**: Medical diagnosis based on symptoms.
  - **Marketing**: Customer segmentation for targeted campaigns.

**Foundational Role**: They serve as the building blocks for more powerful ensemble methods like Random Forests and Gradient Boosting.

### Feature Selection
- Feature selection is a critical step in building a decision tree model, as it determines which features will be used to split the data at each node. The goal is to select the feature that provides the best separation of the classes or values, leading to a more accurate and efficient model. Common criteria for feature selection include:
  - **Gini Impurity**: Measures how often a randomly chosen element would be incorrectly identified if it were randomly labeled according to the distribution of labels in the subset.
  - **Entropy/Information Gain**: Measures the reduction in uncertainty or impurity after a split. The feature that results in the highest information gain is typically selected for splitting.
- > Who selects the feature? The algorithm automatically evaluates all available features at each node and selects the one that provides the best split based on the chosen criterion (Gini Impurity or Information Gain). This process is repeated recursively until the stopping condition is met (e.g., maximum tree depth, minimum samples per leaf, etc.).

#### 1. Information Gain
- Information Gain is a measure of the reduction in uncertainty or impurity after a dataset is split based on a particular feature. It tells us how useful a question or feature is for splitting the data into groups. It is calculated as the difference between the entropy of the parent node and the weighted average entropy of the child nodes. The feature that results in the highest information gain is typically selected for splitting the data at each node in a decision tree.
- **Mathematical Definition**: \
Information Gain (IG) for a feature $A$ is defined as:
$$
IG(S, A) = H(S) - \sum_{v \in Values(A)} \frac{|S_v|}{|S|} H(S_v)
$$
Where:
- $ S $ is the original dataset.
- $ A $ is the feature being evaluated.
- $ Values(A) $ is the set of all possible values for feature $ A $.
- $ S_v $ is the subset of $ S $ for which feature $ A $ has value $ v $.
- $ H(S) $ is the entropy of the original dataset, and $ H(S_v) $ is the entropy of the subset $ S_v $.

For example, in our weather forecasting dataset, there are about 21 features that can be used for splitting the data. The algorithm will evaluate each feature and calculate the information gain for each one. The feature with the highest information gain will be selected for splitting the data at that node in the decision tree.


**Interpretation**: A higher information gain indicates that the feature provides a better split of the data, leading to more homogeneous child nodes. Therefore, the feature with the highest information gain is selected for splitting the data at each node in the decision tree.

## 1. Importing Libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('darkgrid')
plt.rcParams['font.size'] = 14
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.facecolor'] = '#00000000'

## 2. Loading the Dataset

In [ ]:
rain_data = pd.read_csv('../data/weatherAUS.csv')
rain_data.head()

In [ ]:
rain_data.info()

In [ ]:
rain_data.dtypes.value_counts()

In [ ]:
rain_data.isnull().sum()

ways to drop null values in a dataset using dropna() method in pandas:
- `df.dropna()` - This will drop all rows that contain any null values.
- `df.dropna(how='all')` - This will drop only those rows where all values are null.
- `df.dropna(subset=['column_name'])` - This will drop rows where the specified column contains null values.
- `df.dropna(thresh=n)` - This will drop rows that have less than n non-null values.
- `df.dropna(axis=1)` - This will drop columns that contain any null values. \
-> Here `inplace=True` can be used to modify the original DataFrame without creating a new one.

In [ ]:
rain_data.dropna(subset=['RainToday', 'RainTomorrow'], inplace=True)

In [ ]:
rain_data

## 3. Exploratory Data Analysis (EDA) and Visualization

This step is already covered in the **Linear Regression** notebook, so I will skip it here.

## 4. Preparing the Data for Training

Steps to follow to prepare the data for training:
1. Creating a train/validation/test split of the dataset
2. Identifying the input and target features/columns
3. Identifying categorical and numerical columns
4. Imputing (filling) missing numeric values (if any)
5. Scaling the numeric values to the $(0, 1)$ range
6. Encoding the categorical values to one-hot vectors

### 4.1. Train/Validation/Test Split

In [ ]:
year = pd.to_datetime(rain_data.Date).dt.year
sns.countplot(
    x=year
)
plt.title("Number of Rows/Records per Year")
plt.xlabel("Year")
plt.ylabel("Count")
plt.xticks(rotation=45)

plt.show()

In [ ]:
year_counts = pd.to_datetime(rain_data['Date']).dt.year.value_counts().sort_index()
sns.barplot(x=year_counts.index, y=year_counts.values)
plt.title("Number of Rows/Records per Year")
plt.xlabel("Year")
plt.ylabel("Count")
plt.xticks(rotation=45)
for index, value in enumerate(year_counts.values):
    plt.text(index, value, str(value), ha='center', va='bottom')
plt.show()

In [ ]:
year = pd.to_datetime(rain_data.Date).dt.year
train_data = rain_data[year < 2015]
val_data = rain_data[year == 2015]
test_data = rain_data[year > 2015]

Checking shapes of the splits

In [ ]:
print("Train dataset shape: ", train_data.shape)
print("Validation dataset shape: ", val_data.shape)
print("Test dataset shape: ", test_data.shape)

### 4.2. Input and target features/columns and data

In [ ]:
input_cols = list(rain_data.columns)[1:-1]
target_col = 'RainTomorrow'

**Input data and target data**

In [ ]:
train_input_data = train_data[input_cols].copy()
train_target_data = train_data[target_col].copy()

In [ ]:
val_input_data = val_data[input_cols].copy()
val_target_data = val_data[target_col].copy()

In [ ]:
test_input_data = test_data[input_cols].copy()
test_target_data = test_data[target_col].copy()

In [ ]:
train_input_data

In [ ]:
val_input_data

In [ ]:
test_input_data

### 4.3.Categorical and numerical columns

Where should I extract these categorical and numerical columns from? (From training data, validation data, or test data?, If from training data, from which split of the training data (whole training data, or from the input data of the training data)?)
- Categorical columns and numerical columns (features) should be extracted from the training data only, not from the validation or test data. This is because the training data is used to fit the model and determine the relationships between features and the target variable. The validation and test data should be kept separate to evaluate the model's performance on unseen data. Extracting categorical and numerical columns from the training data ensures that the model is trained on the correct feature types and can generalize well to new data.

In [ ]:
numeric_cols = train_input_data.select_dtypes(include=np.number).columns.to_list()
# categorical_cols = train_input_data.select_dtypes(exclude=np.number).columns.to_list()
categorical_cols = train_input_data.select_dtypes(include='str').columns.to_list()

In [ ]:
numeric_cols

### 4.4. Imputing (filling) missing numeric values

In [ ]:
rain_data[numeric_cols].isnull().sum().sort_values(ascending=False)

This shows, except for the `Rainfall` (has $0$ missing values) column, all other numeric columns have missing values.

So I will use `SimpleImputer` from `sklearn` to fill the missing values in the numeric columns with the mean of each column.

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')

In [ ]:
imputer.fit(rain_data[numeric_cols])

In [ ]:
# rain_data[numeric_cols] = imputer.fit(rain_data[numeric_cols])

In [ ]:
# rain_data[numeric_cols].isnull().sum()

In [ ]:
train_input_data[numeric_cols] = imputer.transform(train_input_data[numeric_cols])
val_input_data[numeric_cols] = imputer.transform(val_input_data[numeric_cols])
test_input_data[numeric_cols] = imputer.transform(test_input_data[numeric_cols])

In [ ]:
train_input_data[numeric_cols].isnull().sum()

In [ ]:
val_input_data[numeric_cols].isnull().sum()

In [ ]:
test_input_data[numeric_cols].isnull().sum()

Above I can see that there are no missing numeric values after imputation in all splitted datasets (training, validation, and test datasets).

### 4.5. Scaling the numeric values to the $(0, 1)$ range

For scaling the numeric values to the $(0, 1)$ range, I will use `MinMaxScaler` from `sklearn`. This scaler transforms the features by scaling each feature to a given range, which is typically between 0 and 1. This is done by subtracting the minimum value of the feature and then dividing by the range (max - min) of the feature. Scaling the numeric values helps to ensure that all features contribute equally to the model training and can improve the performance of algorithms that are sensitive to the scale of the data.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

In [ ]:
rain_data[numeric_cols].describe().loc[['max', 'min']]

In [ ]:
scaler.fit(rain_data[numeric_cols])

In [ ]:
train_input_data[numeric_cols] = scaler.transform(train_input_data[numeric_cols])
val_input_data[numeric_cols] = scaler.transform(val_input_data[numeric_cols])
test_input_data[numeric_cols] = scaler.transform(test_input_data[numeric_cols])

In [ ]:
train_input_data[numeric_cols].describe().loc[['min', 'max']]

In [ ]:
train_input_data[numeric_cols]

Let me verify the MinTemp's first 5 values after scaling using `MinMaxScaler` formula:
- Before scaling:
| MinTemp |
|---------|
| 13.4    |
| 7.4     |
| 12.9    |
| 11.9    |
| 8.8     |
- Max value of MinTemp ($X_{max}$): 33.9
- Min value of MinTemp ($X_{min}$): -8.5
- Scaling formula for MinMaxScaler:
$$
X_{scaled} = \frac{X - X_{min}}{X_{max} - X_{min}}
$$
- After scaling:
For first value (13.4):
$$
X_{scaled} = \frac{13.4 - (-8.5)}{33.9 - (-8.5)} = \frac{21.9}{42.4} \approx 0.516509434
$$
For second value (7.4):
$$
X_{scaled} = \frac{7.4 - (-8.5)}{33.9 - (-8.5)} = \frac{15.9}{42.4} \approx 0.375
$$
For third value (12.9):
$$
X_{scaled} = \frac{12.9 - (-8.5)}{33.9 - (-8.5)} = \frac{21.4}{42.4} \approx 0.504716981
$$
For fourth value (11.9):
$$
X_{scaled} = \frac{11.9 - (-8.5)}{33.9 - (-8.5)} = \frac{20.4}{42.4} \approx 0.481132075
$$
For fifth value (8.8):
$$
X_{scaled} = \frac{8.8 - (-8.5)}{33.9 - (-8.5)} = \frac{17.3}{42.4} \approx 0.408018867
$$

- So the first 5 values of MinTemp after scaling are:
| MinTemp (scaled) |
-------------------|
| 0.516509434      |
| 0.375            |
| 0.504716981      |
| 0.481132075      |
| 0.408018867      |

> Here I successfully showed the Math behind the MinMaxScaler and verified the first 5 values of MinTemp after scaling using the formula.

In [ ]:
val_input_data[numeric_cols].describe().loc[['min', 'max']]

In [ ]:
test_input_data[numeric_cols].describe().loc[['min', 'max']]

### 4.6. Encoding the categorical values to one-hot vectors
- For encoding the categorical values to one-hot vectors, I will use `OneHotEncoder` from `sklearn`. This encoder transforms categorical features into a format that can be provided to machine learning algorithms to improve predictions. It does this by creating new binary columns for each category in the original feature. Each column corresponds to a category, and the value is 1 if the original feature matches that category and 0 otherwise. This allows the model to understand the categorical data in a numerical format, which can improve the performance of algorithms that require numerical input.

In [ ]:
categorical_cols

In [ ]:
rain_data[categorical_cols].nunique()

Here:
- Location: 49 unique values
- WindGustDir: 16 unique values
- WindDir9am: 16 unique values
- WindDir3pm: 16 unique values
- RainToday: 2 unique values
-> So there will be 49 + 16 + 16 + 16 + 2 = 99 new binary columns created after one-hot encoding the categorical features. There is a `nan` value in the `WindGustDir`, `WindDir9am`, and `WindDir3pm` columns, so in total there will be 99 + 3 = 102 new binary columns created after one-hot encoding the categorical features. But I will replace `nan` with `Unknown`.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.fit(rain_data[categorical_cols])

In [ ]:
rain_data[categorical_cols].isna().sum()

In [ ]:
rain_data[categorical_cols] = rain_data[categorical_cols].fillna('Unknown').copy()

In [ ]:
rain_data[categorical_cols].isna().sum()

In [ ]:
rain_data[categorical_cols].nunique()

In [ ]:
encoded_cols = list(encoder.get_feature_names_out(categorical_cols))
len(encoded_cols)

In [ ]:
train_input_data[encoded_cols] = encoder.transform(train_input_data[categorical_cols])
val_input_data[encoded_cols] = encoder.transform(val_input_data[categorical_cols])
test_input_data[encoded_cols] = encoder.transform(test_input_data[categorical_cols])

In [ ]:
# # because of PerformanceWarning, better to use the code below
# train_input_data = pd.concat([train_input_data, pd.DataFrame(encoder.transform(train_input_data[categorical_cols]), columns=encoded_cols)], axis=1)
# val_input_data = pd.concat([val_input_data, pd.DataFrame(encoder.transform(val_input_data[categorical_cols]), columns=encoded_cols)], axis=1)
# test_input_data = pd.concat([test_input_data, pd.DataFrame(encoder.transform(test_input_data[categorical_cols]), columns=encoded_cols)], axis=1)

In [ ]:
len(train_input_data[numeric_cols].columns.to_list())

In [ ]:
len(train_input_data[encoded_cols].columns.to_list())

In [ ]:
len(train_input_data.columns.to_list())

Now there are $102 + 16 + 5 = 123$ columns in the training, validation, and test datasets after encoding the categorical features and scaling the numeric features. The 5 columns that are not encoded because they are not categorical are: Location, WindGustDir, WindDir9am, WindDir3pm, RainToday. So I will use only $118$ columns (102 encoded categorical columns + 16 scaled numeric columns - 5 non-encoded columns) for training the Decision Tree and Random Forest models.

**Datasets for training the Decision Tree and Random Forest models**
- For training the Decision Tree and Random Forest models, I will use the input data of the training dataset, which contains the encoded categorical features and scaled numeric features. The target variable will be the `RainTomorrow` column from the training dataset. The validation and test datasets will be used later for evaluating the performance of the trained models.

In [ ]:
# columns for training
total_cols = encoded_cols + numeric_cols
len(total_cols)

In [ ]:
train_input_data[total_cols]

In [ ]:
train_data[target_col]

In [ ]:
val_input_data[total_cols]

In [ ]:
val_data[target_col]

In [ ]:
test_input_data[total_cols]

In [ ]:
test_data[target_col]

In [ ]:
X_train = train_input_data[total_cols]
X_val = val_input_data[total_cols]
X_test = val_input_data[total_cols]

## 5. Saving processed data to disk
- After preparing the data for training, I will save the processed training, validation, and test datasets to disk in parquet format. This allows for efficient storage and retrieval of the data, and it can be easily loaded back into memory for training the models or making predictions in the future. Parquet format is a columnar storage file format that is optimized for performance and can handle large datasets effectively.

In [ ]:
train_input_data.to_parquet('../data/train_datasets/train_inputs.parquet')
val_input_data.to_parquet('../data/val_datasets/val_inputs.parquet')
test_input_data.to_parquet('../data/test_datasets/test_inputs.parquet')

In [ ]:
pd.DataFrame(train_data[target_col]).to_parquet("../data/train_datasets/train_target.parquet")
pd.DataFrame(val_data[target_col]).to_parquet("../data/val_datasets/val_target.parquet")
pd.DataFrame(test_data[target_col]).to_parquet("../data/test_datasets/test_target.parquet")

## 6. Training the Decision Tree model

To train the Decision Tree model, I will use the `DecisionTreeClassifier` from `sklearn`. This classifier is a simple and interpretable model that can be used for classification tasks. I will fit the model on the training data, which includes the encoded categorical features and scaled numeric features, along with the target variable `RainTomorrow`. After training the model, I will evaluate its performance on the validation dataset to see how well it generalizes to unseen data. The evaluation metrics I will use include accuracy, precision, recall, and F1-score.

In [ ]:
X_train = train_input_data[total_cols]
X_target = train_data[target_col]

In [ ]:
train_target_data

In [ ]:
X_target

In [ ]:
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(random_state=42)

In [ ]:
model.fit(X_train, X_target)

### 6.1. Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
train_preds = model.predict(X_train)

In [ ]:
train_preds

In [ ]:
pd.Series(train_preds).value_counts()

In [ ]:
train_target_data.value_counts()

**Probability for each prediction**

In [ ]:
train_probs = model.predict_proba(X_train)
train_probs

_The probability of 1 and 0 tells us that the model is more confident in its predictions when the probabilities are close to 1 or 0, respectively. For example, if the model predicts a probability of 0.9 for class 1 (RainTomorrow = Yes) and 0.1 for class 0 (RainTomorrow = No), it indicates that the model is quite confident that it will rain tomorrow. Conversely, if the probabilities are close to 0.5, it suggests that the model is uncertain about its prediction, as it is almost equally likely to predict either class. Therefore, the closer the probabilities are to 1 or 0, the more confident the model is in its prediction._

In [ ]:
accuracy_score(train_target_data, train_preds)

> The training accuracy of the model is 99.99% (close to 100%), which indicates that the model has learned the patterns in the training data very well (like a student who has memorized the answers to all the questions in a textbook). However, this high training accuracy may also suggest that the model is overfitting the training data, meaning it has learned the noise and specific patterns in the training data rather than generalizing well to unseen data. To confirm whether the model is overfitting, we need to evaluate its performance on the validation dataset. If the validation accuracy is significantly lower than the training accuracy, it would indicate that the model is indeed overfitting.

**Evaluation model on the validation dataset**
- After evaluating the Decision Tree model on the validation dataset, I will calculate the accuracy, precision, recall, and F1-score to assess the model's performance. These metrics will help us understand how well the model is able to predict whether it will rain tomorrow based on the features in the dataset. Accuracy will give us the overall correctness of the model, while precision, recall, and F1-score will provide insights into the model's performance in terms of true positives, false positives, and false negatives, which are crucial for understanding the model's ability to predict the positive class (RainTomorrow = Yes) accurately.

In [ ]:
X_val.head()

In [ ]:
val_preds = model.predict(X_val)

In [ ]:
accuracy_score(val_target_data, val_preds) # 79.5%

Or using `model.score(X_val, val_target_data)` in one line to get the accuracy score.

In [ ]:
model.score(X_val, val_target_data) # 79.5%

Although the training accuracy was very high (close to 100%), the validation accuracy is around 79.5%, which indicates that the model is overfitting the training data. The model has learned the patterns in the training data very well, but it does not generalize well to unseen data, as evidenced by the lower accuracy on the validation dataset. This suggests that the model has learned the noise and specific patterns in the training data rather than capturing the underlying relationships that would allow it to perform well on new data.

### 6.2. Visualization


#### 6.2.1. Visualizing the Decision Tree
- Let me take a look at the decision tree learned from the training data. Visualizing the decision tree can help us understand how the model is making predictions and which features are most important in the decision-making process. It can also help us identify any potential issues with the tree, such as overfitting or underfitting, and provide insights into how we might improve the model through hyperparameter tuning or feature engineering.

In [ ]:
from sklearn.tree import plot_tree, export_text
plt.figure(figsize=(80, 20))
plot_tree(decision_tree=model, feature_names=X_train.columns, max_depth=2, filled=True)
plt.show()

In [ ]:
tree_text = export_text(model, feature_names=list(X_train.columns), max_depth=2)
print(tree_text[:5000])

Here `Humidity3pm` is used as the root node of the decision tree, which means it is the most important feature for predicting whether it will rain tomorrow.
- The question here is how model decided to use `Humidity3pm` as the root node and what is the threshold value of `Humidity3pm` that is used to split the data at the root node.
- To decide which feature to use as the root node, there are different criteria that the decision tree algorithm can use, such as Gini Impurity or Information Gain.

#### 6.2.2. Feature Selection

##### Gini Impurity/Index
- Gini Impurity is a measure of how often a randomly chosen element from the set would be incorrectly labeled if it were randomly labeled according to the distribution of labels in the subset. It is calculated using the formula:
$$ Gini = 1 - \sum_{i=1}^{n} p_i^2 $$
where $p_i$ is the proportion/probability of instances belonging to class $i$ in the subset. 
A Gini Impurity of 0 indicates that all instances belong to a single class (pure), while a Gini Impurity close to 1 indicates a more mixed distribution of classes. The feature that results in the lowest Gini Impurity after the split is typically selected for splitting the data at that node in the decision tree.

In [ ]:
print(train_input_data.shape)
print(val_input_data.shape)
print(test_input_data.shape)

In [ ]:
train_target_data.value_counts()

But the above gini calculation is theoretical. The model actually has not analyzed all possible decisions because that is very inefficient. So there are some techniques which are called **heuristics**. These are strategies to figure out good enough decisions and there are also some randomization/strategies involved there.

In [ ]:
# how deep the tree is
model.tree_.max_depth # 49 layers deep

#### 6.2.3. Feature Importance

Based on the gini index computations, a decision tree assigns an "importance" value to each feature. These values can be used to interpret the results given by a decision tree.

In [ ]:
len(train_input_data[total_cols].columns.to_list())

In [ ]:
len(model.feature_importances_) # for 118 columns, there are 118 feature_importance values

In [ ]:
model.feature_importances_

In [ ]:
feature_importances_df = pd.DataFrame(
    {
        'feature': train_input_data[total_cols].columns,
        'importance': model.feature_importances_
    }
)

**$10$ most important features**

In [ ]:
feature_importances_df.sort_values(by='importance', ascending=False, inplace=True)
feature_importances_df.head(n=10)

In [ ]:
plt.title('Feature Importances')
sns.barplot(
    data=feature_importances_df.head(10),
    x='importance',
    y='feature',
    hue='feature'
)
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

In [ ]:
px.bar(
    data_frame=feature_importances_df.head(10),
    x='feature',
    y='importance',
    title='Feature Importances',
    # how to have with different colors for each
    color='feature'
)

### 6.3. Hyperparameter tuning and Regularization
- One of the most important parts in any ML project is reducing overfitting and improving the generalization of the model. To achieve this, hyperparameter tuning is a crucial step. Hyperparameters are the parameters that are set before the training process begins and are not learned from the data (parameters in Linear models like $weights$ and $bias$). Hyperparameters are configured manually before training. They control the behavior of the learning algorithm and can significantly impact the performance of the model. By tuning hyperparameters, we can find the optimal settings that help reduce overfitting and improve the model's ability to generalize to unseen data. This can be done using techniques such as grid search, random search, or Bayesian optimization, which systematically explore different combinations of hyperparameters to find the best configuration for the model. The process of reducing overfitting is $Regularization$.

Hyperparameters are arguments passed to `DecisionTreeClassifier()`.

#### -> Using `max_depth` hyperparameter

In [ ]:
DecisionTreeClassifier()

In [ ]:
model.tree_.max_depth # having 49 layers is the reason for overfitting

In [ ]:
model = DecisionTreeClassifier(max_depth=3, random_state=42)

In [ ]:
model.fit(X=X_train, y=X_target)

In [ ]:
model.score(X_train, train_target_data) # 83.16%

In [ ]:
model.score(X_val, val_target_data) # 83.48%

In [ ]:
model.classes_

In [ ]:
plt.figure(figsize=(80, 20))
plot_tree(decision_tree=model, feature_names=list(X_train.columns), max_depth=3, filled=True, rounded=True, class_names=model.classes_)
plt.show()

In [ ]:
print(export_text(model, feature_names=list(X_train.columns)))

How to know best value for `max_depth`? I will experiment with different values of `max_depth` here to know best value.

In [ ]:
val_input_data

In [ ]:
def max_depth_errors(md: int):
    model = DecisionTreeClassifier(max_depth=md, random_state=42)
    model.fit(X_train, X_target)
    train_acc_err = 1 - model.score(X_train, X_target)
    val_acc_err = 1 - model.score(X_val, val_target_data)
    return {'Max Depth': md, 'Training error': train_acc_err, 'Validation error': val_acc_err}

In [ ]:
# dataframe made from list of dictionaries: {'Max_depth': md, 'Training error: 'training_acc_err, 'Validation_error': 'val_acc_err'}
 # recommended range for max_depth values is 1 up to 21
error_df = pd.DataFrame([max_depth_errors(md) for md in range(1, 21)])

In [ ]:
error_df

In [ ]:
plt.figure()
plt.plot(error_df['Max Depth'], error_df['Training error'])
plt.plot(error_df['Max Depth'], error_df['Validation error'])
plt.xlabel('Max Depth')
plt.ylabel('Prediction Error(1 - Accuracy)')
plt.legend(['Max Depth vs Training Error', 'Max Depth vs Validation Error'])
plt.xticks(range(0, 21, 2), rotation='vertical')
plt.title('Training vs Validation Error')
plt.show()

> NOTE: As the `max_depth` value increases, model starts to memorize the data rather than generalizing it. That is why training accuracy is increasing (error is decreasing) and validation accuracy is decreasing (erro is increasing).

![Accuracy](../data/img.png)

We stop training model at the point validation error line starts to change direction (Best Fit in above figure). That is where `max_depth` should be taken. In our case, it changes direction (decreasing to increasing) at $max\_ depth=7$: (From **0.154661** ($6$) → **0.15390** ($7$) → **154427** ($8$)).

**The final best trained model with low error is at `max_depth = 7`**.
- This done is by using `max_depth` hyperparameter.

In [ ]:
model = DecisionTreeClassifier(max_depth=7, random_state=42).fit(X_train, X_target)
model.score(X_train, X_target), model.score(X_val, val_target_data) # Accuracy is 85.1%

#### -> Using `max_leaf_nodes` hyperparameter

In [ ]:
model = DecisionTreeClassifier(max_leaf_nodes=128, random_state=42)

In [ ]:
model.fit(X_train, X_target)

In [ ]:
model.score(X_train, X_target), model.score(X_val, val_target_data)

In [ ]:
model.tree_.max_depth

In [ ]:
model_text = export_text(model, feature_names=list(X_train.columns))
print(model_text[:3000])

There are different level of paths. The largest one is 11.

> There is another more advanced technique (but less commonly used) is **$Cost\ Complexity\ Pruning$**.

---